# Quantum Approximate Optimization Algorithm (QAOA)

This notebook demonstrates the implementation of QAOA for the Max-Cut problem using Qiskit.

## Problem Definition
Max-Cut is a combinatorial optimization problem where the goal is to partition the vertices of a graph into two sets such that the number of edges between the sets is maximized.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter


def create_qaoa_circ(graph, theta):
    """
    Creates a simple QAOA circuit for Max-Cut.
    graph: list of tuples representing edges (e.g., [(0, 1), (1, 2)])
    theta: list of parameters [gamma, beta]
    """
    nqubits = len(set([v for edge in graph for v in edge]))
    qc = QuantumCircuit(nqubits)

    gamma = theta[0]
    beta = theta[1]

    # Initial state: superposition
    for i in range(nqubits):
        qc.h(i)

    # Cost Hamiltonian (ZZ interactions for Max-Cut)
    for edge in graph:
        qc.rzz(2 * gamma, edge[0], edge[1])

    # Mixer Hamiltonian (X rotations)
    for i in range(nqubits):
        qc.rx(2 * beta, i)

    qc.measure_all()
    return qc


# Example: Max-Cut on a 3-node cycle graph
graph = [(0, 1), (1, 2), (2, 0)]
# Random initial parameters
theta = [1.0, 1.0]

circuit = create_qaoa_circ(graph, theta)
print("QAOA Circuit Created:")
print(circuit.draw(output="text"))

## Parameter Optimization
We use a classical optimizer to find the best parameters $\gamma$ and $\beta$.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from scipy.optimize import minimize


def get_qaoa_circuit(graph, theta):
    """Constructs the QAOA circuit for Max-Cut."""
    nodes = sorted(list(set([v for edge in graph for v in edge])))
    n_qubits = len(nodes)
    node_to_qubit = {node: i for i, node in enumerate(nodes)}
    qc = QuantumCircuit(n_qubits)

    # Parameters
    gamma, beta = theta[0], theta[1]

    # Initial state: Superposition
    qc.h(range(n_qubits))

    # Cost Hamiltonian (ZZ interactions)
    for edge in graph:
        qc.rzz(2 * gamma, node_to_qubit[edge[0]], node_to_qubit[edge[1]])

    # Mixer Hamiltonian (X rotations)
    qc.rx(2 * beta, range(n_qubits))

    qc.measure_all()
    return qc


def maxcut_cost(bits, graph):
    """Calculates the cut size for a given bitstring."""
    nodes = sorted(list(set([v for edge in graph for v in edge])))
    node_to_qubit = {node: i for i, node in enumerate(nodes)}
    cost = 0
    for edge in graph:
        if bits[node_to_qubit[edge[0]]] != bits[node_to_qubit[edge[1]]]:
            cost -= 1  # We minimize -cost to maximize the cut
    return cost


def objective_function(theta, graph, sampler):
    """Objective function to be minimized."""
    qc = get_qaoa_circuit(graph, theta)

    # Run the circuit
    job = sampler.run([qc])
    result = job.result()[0]
    counts = result.data.meas.get_counts()

    # Calculate expectation value
    avg_cost = 0
    total_shots = sum(counts.values())
    for bitstring, count in counts.items():
        bits = [int(b) for b in bitstring[::-1]]
        avg_cost += maxcut_cost(bits, graph) * count

    return avg_cost / total_shots


# 1. Generate a random graph
n_nodes = 5
prob = 0.5
G = nx.erdos_renyi_graph(n_nodes, prob, seed=42)
graph = list(G.edges())

# 2. Display the graph
plt.figure(figsize=(6, 4))
pos = nx.spring_layout(G)
nx.draw(
    G, pos, with_labels=True, node_color="lightblue", edge_color="gray", node_size=500
)
plt.title("Random 5-Node Graph for Max-Cut")
plt.savefig("images/graph.png")
plt.show()

# 3. Setup Sampler and Optimizer
sampler = StatevectorSampler()
init_params = [1.0, 1.0]

# 4. Optimize
print("Optimizing QAOA parameters...")
res = minimize(objective_function, init_params, args=(graph, sampler), method="COBYLA")

print(f"Optimal parameters: gamma={res.x[0]:.4f}, beta={res.x[1]:.4f}")